# Transfer Learning on YOLOv11n — Feature Extraction + Fine-Tuning

## Two-Stage Transfer Learning Pipeline

This notebook demonstrates the **professional transfer learning technique** applied to YOLOv11n for steel surface defect detection on NEU-DET (6 classes).

### Stage 1: Feature Extraction
- Load YOLOv11n pretrained on COCO (80 classes)
- **Freeze ALL backbone + neck layers** (keep pretrained weights intact)
- Replace detection head for **6 classes** (NEU-DET)
- Train **only the detection head** to learn defect-specific features
- The pretrained feature extractor stays unchanged

### Stage 2: Fine-Tuning
- Load the best weights from Stage 1
- **Unfreeze the top layers** (neck + head) — these learn high-level features
- **Keep early layers frozen** (backbone) — these learn generic low-level features (edges, textures)
- Use a **very low learning rate** to avoid destroying pretrained weights
- Fine-tune to improve defect-specific detection

### Layer Distribution (YOLOv11n — 181 layers)
```
BACKBONE (Layers 0-10):  ~90 layers  — Low-level features (edges, textures, gradients)
NECK     (Layers 11-22): ~75 layers  — Mid-level features (multi-scale fusion)
HEAD     (Layer 23):     ~16 layers  — High-level features (classification, bbox)
```

### Why This Works
- COCO pretrained features (edges, textures, shapes) are **universal** — they transfer well to steel defects
- Only the head needs to learn: "which patterns = which defect class"
- Fine-tuning the neck helps the model adapt multi-scale fusion to defect sizes
- Early backbone layers stay frozen because edge/texture detection is already optimal

In [ ]:
# ============================================================
# Cell 1: Setup & Configuration
# ============================================================
import torch
import torch.nn as nn
from ultralytics import YOLO
from pathlib import Path
import json
import matplotlib.pyplot as plt

# ── Reproducibility ──
SEED = 42
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# ── Paths ──
ROOT = Path(r"D:\DigiSteel-Yolo\DigiSteel-YOLO")
DATA_YAML = ROOT / "configs" / "data" / "neu_det.yaml"
COCO_WEIGHTS = "yolo11n.pt"  # Pretrained on COCO (80 classes)

# Stage 1 output
STAGE1_PROJECT = ROOT / "runs" / "transfer_learning"
STAGE1_NAME = "stage1_feature_extraction"
STAGE1_WEIGHTS = STAGE1_PROJECT / STAGE1_NAME / "weights" / "best.pt"

# Stage 2 output
STAGE2_NAME = "stage2_fine_tuning"
STAGE2_WEIGHTS = STAGE1_PROJECT / STAGE2_NAME / "weights" / "best.pt"

# ── Hyperparameters ──
NUM_CLASSES = 6       # NEU-DET: crazing, inclusion, patches, pitted_surface, rolled-in_scale, scratches
IMGSZ = 800           # Image size (upscaled from 200x200 native)
BATCH = 8             # Batch size (RTX 2000 Ada: 17GB VRAM)
DEVICE = 0 if torch.cuda.is_available() else "cpu"

# ── Verify ──
print("=" * 60)
print("  TRANSFER LEARNING SETUP")
print("=" * 60)
print(f"\n  Device:       {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")
print(f"  CUDA:         {torch.cuda.is_available()}")
print(f"  Num classes:  {NUM_CLASSES}")
print(f"  Image size:   {IMGSZ}")
print(f"  Batch size:   {BATCH}")
print(f"  Seed:         {SEED}")
print(f"\n  Data YAML:    {DATA_YAML} (exists: {DATA_YAML.exists()})")
print(f"  COCO weights: {COCO_WEIGHTS}")
print(f"  Stage 1 out:  {STAGE1_WEIGHTS}")
print(f"  Stage 2 out:  {STAGE2_WEIGHTS}")
print("\n" + "=" * 60)

## Understanding YOLOv11n Architecture

Before we freeze/unfreeze layers, let's understand the model structure:

```
Layer  Type        What It Learns              Freezing Strategy
─────────────────────────────────────────────────────────────────────
0      Conv        Edge detection (3→16)        FREEZE (universal)
1      Conv        Texture patterns (16→32)     FREEZE (universal)
2      C3k2        Simple features (P2)         FREEZE (universal)
3      Conv        Downsample (32→64)           FREEZE (universal)
4      C3k2        Medium features (P3)         FREEZE (universal)
5      Conv        Downsample (64→128)          FREEZE (universal)
6      C3k2        Complex features (P4)        FREEZE (universal)
7      Conv        Downsample (128→256)         FREEZE (universal)
8      C3k2        High-level features (P5)     FREEZE or FINE-TUNE
9      SPPF        Multi-scale pooling          FREEZE or FINE-TUNE
10     C2PSA       Attention mechanism           FREEZE or FINE-TUNE
────── NECK (PAN-FPN) ──────────────────────────────────────────────
11-22  C3k2+Conv   Multi-scale fusion           FINE-TUNE (Stage 2)
────── HEAD ──────────────────────────────────────────────────────
23     Detect      Classification + BBox         ALWAYS TRAIN
```

### Key Insight
- **Layers 0-7** learn **universal features** (edges, textures) — same for any image domain
- **Layers 8-10** learn **domain-specific features** — may need adaptation
- **Layers 11-22** learn **how to combine features** at different scales — needs adaptation for defect sizes
- **Layer 23** learns **what to detect** — MUST be retrained for our 6 classes

In [ ]:
# ============================================================
# Cell 2: Inspect Pretrained Model Architecture
# ============================================================
print("=" * 70)
print("  COCO PRETRAINED MODEL — LAYER INSPECTION")
print("=" * 70)

# Load COCO pretrained model
coco_model = YOLO(COCO_WEIGHTS)

# Print model info
print("\n[1] Model Summary:\n")
coco_model.info(verbose=True)

# List all layers with their index, type, and parameter count
print("\n[2] Layer Table:\n")
print(f"{'Idx':<5} {'Type':<20} {'Params':>12} {'Frozen?':>10} {'Feature Level'}")
print("-" * 70)

layer_list = list(coco_model.model.model.children())
total_params = sum(p.numel() for p in coco_model.model.parameters())

for i, layer in enumerate(layer_list):
    layer_type = type(layer).__name__
    num_params = sum(p.numel() for p in layer.parameters())
    
    # Feature level
    if i <= 7:
        level = "LOW (edges, textures)"
    elif i <= 10:
        level = "MID (complex patterns)"
    elif i <= 22:
        level = "MID-HIGH (fusion)"
    else:
        level = "HIGH (classification)"
    
    print(f"{i:<5} {layer_type:<20} {num_params:>12,} {'---':>10} {level}")

print("-" * 70)
print(f"{'TOTAL':<5} {'':<20} {total_params:>12,}")

# Show COCO detection head (80 classes)
print("\n[3] COCO Detection Head (80 classes):\n")
detect_layer = layer_list[-1]
print(f"  Type:        {type(detect_layer).__name__}")
print(f"  nc (classes): {detect_layer.nc}")
print(f"  Reg_max:     {detect_layer.reg_max}")
print(f"  Anchors:     {detect_layer.no} per anchor")
print(f"  Parameters:  {sum(p.numel() for p in detect_layer.parameters()):,}")

print("\n" + "=" * 70)

---

## Stage 1: Feature Extraction

**Goal:** Train ONLY the detection head while keeping all pretrained weights frozen.

### What Happens
1. Load YOLOv11n pretrained on COCO (80 classes)
2. Freeze ALL backbone + neck layers (layers 0-22)
3. Replace detection head: 80 classes → 6 classes
4. Train with standard learning rate (only head updates)
5. The model learns: "which patterns = which steel defect class"

### Why Freeze?
- COCO pretrained features (edges, textures, shapes) are **universal**
- Training the whole model on 1,290 images would **overfit**
- Only the head needs to learn the new class mapping
- This is the **fastest** and **safest** transfer learning approach

In [ ]:
# ============================================================
# Cell 3: Stage 1 — Feature Extraction (Freeze All + Train Head)
# ============================================================
print("=" * 70)
print("  STAGE 1: FEATURE EXTRACTION")
print("  Freeze ALL layers → Train ONLY detection head")
print("=" * 70)

# ── Step 1: Load pretrained model ──
print("\n[Step 1] Loading COCO pretrained YOLOv11n...")
model = YOLO(COCO_WEIGHTS)
print(f"  ✓ Loaded: {COCO_WEIGHTS}")
print(f"  ✓ Pretrained on COCO: 80 classes")
print(f"  ✓ Total parameters: {sum(p.numel() for p in model.model.parameters()):,}")

# ── Step 2: Freeze ALL backbone + neck layers ──
print("\n[Step 2] Freezing ALL backbone + neck layers...")

frozen_count = 0
trainable_count = 0

for name, param in model.model.named_parameters():
    # Detect layer should NOT be frozen (we need to retrain it for 6 classes)
    if "model.23" in name or "detect" in name.lower():
        param.requires_grad = True
        trainable_count += 1
    else:
        param.requires_grad = False
        frozen_count += 1

print(f"  ✓ Frozen parameters:     {frozen_count}")
print(f"  ✓ Trainable parameters:  {trainable_count} (detection head only)")

# ── Step 3: Verify freezing ──
print("\n[Step 3] Verifying layer freezing status:\n")
print(f"{'Layer':<15} {'Type':<20} {'Params':>12} {'Status':>10}")
print("-" * 60)

for i, layer in enumerate(model.model.model.children()):
    layer_type = type(layer).__name__
    num_params = sum(p.numel() for p in layer.parameters())
    is_frozen = all(not p.requires_grad for p in layer.parameters()) if list(layer.parameters()) else True
    status = "🔒 FROZEN" if is_frozen else "🔥 TRAINABLE"
    print(f"  Layer {i:<3} {layer_type:<20} {num_params:>12,} {status:>10}")

# ── Step 4: Show the new detection head ──
print("\n[Step 4] Detection Head (will be retrained for 6 classes):\n")
detect = model.model.model[-1]
print(f"  Old classes (COCO): 80")
print(f"  New classes (NEU):  {NUM_CLASSES}")
print(f"  Head parameters:    {sum(p.numel() for p in detect.parameters()):,}")
print(f"  Status:             🔥 TRAINABLE")

print("\n" + "=" * 70)
print("  Ready to train Stage 1 (feature extraction)")
print("=" * 70)

In [ ]:
# ============================================================
# Cell 4: Stage 1 — Train (Feature Extraction)
# ============================================================
print("=" * 70)
print("  STAGE 1: TRAINING (Feature Extraction)")
print("  Only detection head is trained")
print("=" * 70)

# Check if Stage 1 weights already exist
if STAGE1_WEIGHTS.exists():
    print(f"\n✓ Stage 1 weights already exist: {STAGE1_WEIGHTS}")
    print("  Skipping training. Delete the folder to retrain.")
    stage1_results = None
else:
    print("\nTraining Stage 1...")
    print(f"  Epochs: 50 (fewer epochs — only training head)")
    print(f"  LR0:    0.01 (standard — head needs to learn fast)")
    print(f"  Patience: 20")
    
    # Train with frozen backbone
    # Ultralytics handles the head replacement automatically when nc differs
    stage1_results = model.train(
        data=str(DATA_YAML),
        imgsz=IMGSZ,
        batch=BATCH,
        epochs=50,              # Fewer epochs — only training head
        patience=20,            # Early stopping
        lr0=0.01,               # Standard LR — head needs to learn
        lrf=0.01,               # Final LR = initial (no decay for head-only)
        cos_lr=False,           # No cosine — constant LR for head
        close_mosaic=10,        # Disable mosaic in last 10 epochs
        project=str(STAGE1_PROJECT),
        name=STAGE1_NAME,
        seed=SEED,
        pretrained=True,
        optimizer="auto",       # Default optimizer
        verbose=True,
        plots=True,
        exist_ok=True,
    )
    
    print(f"\n✓ Stage 1 training complete!")
    print(f"  Best weights: {STAGE1_WEIGHTS}")

print("\n" + "=" * 70)

In [ ]:
# ============================================================
# Cell 5: Stage 1 — Evaluation
# ============================================================
print("=" * 70)
print("  STAGE 1: EVALUATION")
print("=" * 70)

# Load best Stage 1 weights
if STAGE1_WEIGHTS.exists():
    print(f"\nLoading Stage 1 best weights: {STAGE1_WEIGHTS}")
    stage1_model = YOLO(str(STAGE1_WEIGHTS))
    
    # Evaluate on test set
    print("\nEvaluating on test set...")
    metrics = stage1_model.val(
        split='test',
        data=str(DATA_YAML),
        imgsz=IMGSZ,
        batch=BATCH,
        verbose=True
    )
    
    # Print results
    print("\n" + "-" * 60)
    print("STAGE 1 RESULTS (Feature Extraction)")
    print("-" * 60)
    print(f"  mAP@0.5:      {metrics.box.map50:.4f} ({metrics.box.map50*100:.1f}%)")
    print(f"  mAP@0.5:0.95: {metrics.box.map:.4f} ({metrics.box.map*100:.1f}%)")
    print(f"  Precision:    {metrics.box.mp:.4f} ({metrics.box.mp*100:.1f}%)")
    print(f"  Recall:       {metrics.box.mr:.4f} ({metrics.box.mr*100:.1f}%)")
    
    # Per-class
    print("\n  Per-class mAP@0.5:")
    import yaml
    with open(DATA_YAML) as f:
        names = yaml.safe_load(f).get('names', {})
    if isinstance(names, list):
        names = {i: n for i, n in enumerate(names)}
    
    for idx, (cid, cname) in enumerate(sorted(names.items())):
        if idx < len(metrics.box.maps):
            val = metrics.box.maps[idx]
            bar = '█' * int(val * 40)
            print(f"    {cname:<18} {val*100:.1f}% {bar}")
    
    # Save results
    stage1_metrics = {
        "stage": "feature_extraction",
        "mAP50": float(metrics.box.map50),
        "mAP50_95": float(metrics.box.map),
        "precision": float(metrics.box.mp),
        "recall": float(metrics.box.mr),
        "per_class": {names[i]: float(metrics.box.maps[i]) for i in range(len(names)) if i < len(metrics.box.maps)}
    }
    results_path = STAGE1_PROJECT / "stage1_results.json"
    with open(results_path, "w") as f:
        json.dump(stage1_metrics, f, indent=2)
    print(f"\n  Results saved: {results_path}")
else:
    print(f"\n✗ Stage 1 weights not found: {STAGE1_WEIGHTS}")
    print("  Run Cell 4 first.")

print("\n" + "=" * 70)

---

## Stage 2: Fine-Tuning

**Goal:** Unfreeze the neck + head layers and fine-tune with a low learning rate.

### What Happens
1. Load best weights from Stage 1
2. **Keep early backbone frozen** (layers 0-7) — universal edge/texture features
3. **Unfreeze late backbone + neck + head** (layers 8-23) — domain-specific features
4. Use **very low learning rate** (0.001) to avoid destroying pretrained weights
5. Train for more epochs to let the model adapt to steel defects

### Why Unfreeze the Neck?
- The neck learns **how to combine features** at different scales
- Steel defects vary in size: crazing (tiny) vs patches (large)
- COCO objects have different size distributions than steel defects
- Fine-tuning the neck helps the model adapt multi-scale fusion

### Why Keep Early Layers Frozen?
- Layers 0-7 learn **universal features** (edges, textures, gradients)
- These features are the same for ANY image domain
- Unfreezing them risks **catastrophic forgetting** with limited data
- 1,290 training images is too few to retrain from scratch

In [ ]:
# ============================================================
# Cell 6: Stage 2 — Fine-Tuning (Unfreeze Top Layers)
# ============================================================
print("=" * 70)
print("  STAGE 2: FINE-TUNING")
print("  Freeze early layers → Unfreeze top layers → Low LR")
print("=" * 70)

# ── Step 1: Load Stage 1 best weights ──
print("\n[Step 1] Loading Stage 1 best weights...")
if not STAGE1_WEIGHTS.exists():
    print(f"  ✗ Stage 1 weights not found: {STAGE1_WEIGHTS}")
    print("  Run Cell 4 first.")
else:
    model_ft = YOLO(str(STAGE1_WEIGHTS))
    print(f"  ✓ Loaded: {STAGE1_WEIGHTS}")
    
    # ── Step 2: Define freeze/unfreeze boundary ──
    # Layers 0-7:  FROZEN  (universal low-level features)
    # Layers 8-23: TRAINABLE (domain-specific + neck + head)
    FREEZE_UP_TO = 7  # Freeze layers 0-7 (inclusive)
    
    print(f"\n[Step 2] Setting freeze boundary at layer {FREEZE_UP_TO}...")
    print(f"  FROZEN:    Layers 0-{FREEZE_UP_TO}  (backbone early — edges, textures)")
    print(f"  TRAINABLE: Layers {FREEZE_UP_TO+1}-23 (backbone late + neck + head)")
    
    # ── Step 3: Apply freezing ──
    print("\n[Step 3] Applying freeze/unfreeze...")
    
    frozen_params = 0
    trainable_params = 0
    
    for name, param in model_ft.model.named_parameters():
        # Extract layer index from parameter name (e.g., "model.5.0.conv.weight" → 5)
        parts = name.split(".")
        if parts[0] == "model" and len(parts) > 1:
            try:
                layer_idx = int(parts[1])
            except ValueError:
                layer_idx = -1
            
            if layer_idx <= FREEZE_UP_TO:
                param.requires_grad = False
                frozen_params += param.numel()
            else:
                param.requires_grad = True
                trainable_params += param.numel()
        else:
            # Non-layer parameters (e.g., detect head)
            param.requires_grad = True
            trainable_params += param.numel()
    
    print(f"  ✓ Frozen:     {frozen_params:>12,} parameters")
    print(f"  ✓ Trainable:  {trainable_params:>12,} parameters")
    print(f"  ✓ Total:      {frozen_params + trainable_params:>12,} parameters")
    print(f"  ✓ Frozen %:   {frozen_params/(frozen_params+trainable_params)*100:.1f}%")
    
    # ── Step 4: Verify layer status ──
    print("\n[Step 4] Layer status:\n")
    print(f"  {'Idx':<5} {'Type':<20} {'Params':>12} {'Status':>10}")
    print(f"  {'-'*50}")
    
    for i, layer in enumerate(model_ft.model.model.children()):
        layer_type = type(layer).__name__
        num_params = sum(p.numel() for p in layer.parameters())
        is_frozen = all(not p.requires_grad for p in layer.parameters()) if list(layer.parameters()) else True
        status = "🔒 FROZEN" if is_frozen else "🔥 TRAINABLE"
        print(f"  {i:<5} {layer_type:<20} {num_params:>12,} {status:>10}")
    
    print("\n" + "=" * 70)
    print("  Ready to train Stage 2 (fine-tuning)")
    print("=" * 70)

In [ ]:
# ============================================================
# Cell 7: Stage 2 — Train (Fine-Tuning with Low LR)
# ============================================================
print("=" * 70)
print("  STAGE 2: TRAINING (Fine-Tuning)")
print("  Frozen: backbone early (0-7) | Trainable: neck + head (8-23)")
print("=" * 70)

# Check if Stage 2 weights already exist
if STAGE2_WEIGHTS.exists():
    print(f"\n✓ Stage 2 weights already exist: {STAGE2_WEIGHTS}")
    print("  Skipping training. Delete the folder to retrain.")
else:
    print("\nTraining Stage 2...")
    print(f"  Epochs:    100 (more epochs — fine-tuning neck + head)")
    print(f"  LR0:       0.001 (LOW — 10x smaller than Stage 1)")
    print(f"  LRF:       0.0001 (cosine decay to 0.0001)")
    print(f"  Cosine LR: True (gradual decay)")
    print(f"  Patience:  30")
    
    # Fine-tune with low learning rate
    stage2_results = model_ft.train(
        data=str(DATA_YAML),
        imgsz=IMGSZ,
        batch=BATCH,
        epochs=100,             # More epochs for fine-tuning
        patience=30,            # More patience
        lr0=0.001,              # LOW LR — 10x smaller than Stage 1
        lrf=0.0001,             # Cosine decay to 0.0001
        cos_lr=True,            # Cosine schedule for gradual decay
        close_mosaic=15,        # Disable mosaic in last 15 epochs
        project=str(STAGE1_PROJECT),  # Same project folder
        name=STAGE2_NAME,
        seed=SEED,
        pretrained=False,       # Don't reload pretrained — use our Stage 1 weights
        optimizer="auto",
        verbose=True,
        plots=True,
        exist_ok=True,
    )
    
    print(f"\n✓ Stage 2 fine-tuning complete!")
    print(f"  Best weights: {STAGE2_WEIGHTS}")

print("\n" + "=" * 70)

In [ ]:
# ============================================================
# Cell 8: Stage 2 — Evaluation
# ============================================================
print("=" * 70)
print("  STAGE 2: EVALUATION")
print("=" * 70)

# Load best Stage 2 weights
if STAGE2_WEIGHTS.exists():
    print(f"\nLoading Stage 2 best weights: {STAGE2_WEIGHTS}")
    stage2_model = YOLO(str(STAGE2_WEIGHTS))
    
    # Evaluate on test set
    print("\nEvaluating on test set...")
    metrics2 = stage2_model.val(
        split='test',
        data=str(DATA_YAML),
        imgsz=IMGSZ,
        batch=BATCH,
        verbose=True
    )
    
    # Print results
    print("\n" + "-" * 60)
    print("STAGE 2 RESULTS (Fine-Tuning)")
    print("-" * 60)
    print(f"  mAP@0.5:      {metrics2.box.map50:.4f} ({metrics2.box.map50*100:.1f}%)")
    print(f"  mAP@0.5:0.95: {metrics2.box.map:.4f} ({metrics2.box.map*100:.1f}%)")
    print(f"  Precision:    {metrics2.box.mp:.4f} ({metrics2.box.mp*100:.1f}%)")
    print(f"  Recall:       {metrics2.box.mr:.4f} ({metrics2.box.mr*100:.1f}%)")
    
    # Per-class
    print("\n  Per-class mAP@0.5:")
    import yaml
    with open(DATA_YAML) as f:
        names = yaml.safe_load(f).get('names', {})
    if isinstance(names, list):
        names = {i: n for i, n in enumerate(names)}
    
    for idx, (cid, cname) in enumerate(sorted(names.items())):
        if idx < len(metrics2.box.maps):
            val = metrics2.box.maps[idx]
            bar = '█' * int(val * 40)
            print(f"    {cname:<18} {val*100:.1f}% {bar}")
    
    # Save results
    stage2_metrics = {
        "stage": "fine_tuning",
        "mAP50": float(metrics2.box.map50),
        "mAP50_95": float(metrics2.box.map),
        "precision": float(metrics2.box.mp),
        "recall": float(metrics2.box.mr),
        "per_class": {names[i]: float(metrics2.box.maps[i]) for i in range(len(names)) if i < len(metrics2.box.maps)}
    }
    results_path = STAGE1_PROJECT / "stage2_results.json"
    with open(results_path, "w") as f:
        json.dump(stage2_metrics, f, indent=2)
    print(f"\n  Results saved: {results_path}")
else:
    print(f"\n✗ Stage 2 weights not found: {STAGE2_WEIGHTS}")
    print("  Run Cell 7 first.")

print("\n" + "=" * 70)

---

## Comparison: Stage 1 vs Stage 2 vs From Scratch

In [ ]:
# ============================================================
# Cell 9: Compare Stage 1 vs Stage 2
# ============================================================
print("=" * 70)
print("  COMPARISON: STAGE 1 vs STAGE 2")
print("=" * 70)

# Load results
stage1_path = STAGE1_PROJECT / "stage1_results.json"
stage2_path = STAGE1_PROJECT / "stage2_results.json"

s1 = json.load(open(stage1_path)) if stage1_path.exists() else None
s2 = json.load(open(stage2_path)) if stage2_path.exists() else None

if s1 and s2:
    # Overall comparison
    print("\n  {'Metric':<20} {'Stage 1':<15} {'Stage 2':<15} {'Improvement':<15}")
    print("  " + "-" * 65)
    
    metrics_to_compare = [
        ("mAP@0.5", "mAP50"),
        ("mAP@0.5:0.95", "mAP50_95"),
        ("Precision", "precision"),
        ("Recall", "recall"),
    ]
    
    for label, key in metrics_to_compare:
        v1 = s1[key] * 100
        v2 = s2[key] * 100
        diff = v2 - v1
        sign = "+" if diff > 0 else ""
        print(f"  {label:<20} {v1:.1f}%{'':<10} {v2:.1f}%{'':<10} {sign}{diff:.1f}%")
    
    # Per-class comparison
    print("\n  Per-class mAP@0.5:")
    print(f"  {'Class':<20} {'Stage 1':<12} {'Stage 2':<12} {'Change':<10}")
    print("  " + "-" * 54)
    
    for cname in sorted(s1["per_class"].keys()):
        v1 = s1["per_class"][cname] * 100
        v2 = s2["per_class"][cname] * 100
        diff = v2 - v1
        sign = "+" if diff > 0 else ""
        print(f"  {cname:<20} {v1:.1f}%{'':<7} {v2:.1f}%{'':<7} {sign}{diff:.1f}%")
    
    # Visual comparison
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Overall metrics bar chart
    labels = ['mAP@0.5', 'mAP@0.5:0.95', 'Precision', 'Recall']
    s1_vals = [s1['mAP50']*100, s1['mAP50_95']*100, s1['precision']*100, s1['recall']*100]
    s2_vals = [s2['mAP50']*100, s2['mAP50_95']*100, s2['precision']*100, s2['recall']*100]
    
    x = range(len(labels))
    width = 0.35
    
    axes[0].bar([i - width/2 for i in x], s1_vals, width, label='Stage 1 (Feature Extraction)', color='#3498db')
    axes[0].bar([i + width/2 for i in x], s2_vals, width, label='Stage 2 (Fine-Tuning)', color='#e74c3c')
    axes[0].set_ylabel('Score (%)')
    axes[0].set_title('Stage 1 vs Stage 2 — Overall Metrics', fontweight='bold')
    axes[0].set_xticks(x)
    axes[0].set_xticklabels(labels)
    axes[0].legend()
    axes[0].grid(True, alpha=0.3, axis='y')
    
    # Per-class bar chart
    classes = sorted(s1['per_class'].keys())
    s1_class = [s1['per_class'][c]*100 for c in classes]
    s2_class = [s2['per_class'][c]*100 for c in classes]
    
    x2 = range(len(classes))
    axes[1].bar([i - width/2 for i in x2], s1_class, width, label='Stage 1', color='#3498db')
    axes[1].bar([i + width/2 for i in x2], s2_class, width, label='Stage 2', color='#e74c3c')
    axes[1].set_ylabel('mAP@0.5 (%)')
    axes[1].set_title('Stage 1 vs Stage 2 — Per-Class', fontweight='bold')
    axes[1].set_xticks(x2)
    axes[1].set_xticklabels(classes, rotation=45, ha='right')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3, axis='y')
    
    plt.tight_layout()
    plt.show()

else:
    print("\n  Results not found. Run Cell 5 and Cell 8 first.")

print("\n" + "=" * 70)

---

## Summary: Transfer Learning Pipeline

### What We Did

| Stage | What | Frozen | Learning Rate | Epochs |
|-------|------|--------|---------------|--------|
| **1: Feature Extraction** | Load COCO pretrained → freeze all → train head only | All except head | 0.01 | 50 |
| **2: Fine-Tuning** | Load Stage 1 → unfreeze neck → low LR | Backbone early (0-7) | 0.001 | 100 |

### Why This Works

```
Stage 1: "Learn WHAT to detect" (head only)
  → COCO features are universal (edges, textures, shapes)
  → Only the head needs to map: pattern → defect class
  → Fast training, low risk of overfitting

Stage 2: "Learn HOW to detect" (neck + head)
  → Neck adapts multi-scale fusion to defect sizes
  → Low LR preserves pretrained knowledge
  → More epochs for gradual adaptation
```

### Key Hyperparameters

| Parameter | Stage 1 | Stage 2 | Why |
|-----------|---------|---------|-----|
| `lr0` | 0.01 | 0.001 | Stage 2 uses 10x lower LR to preserve weights |
| `lrf` | 0.01 | 0.0001 | Stage 2 decays to very low LR |
| `cos_lr` | False | True | Stage 2 uses cosine for gradual decay |
| `epochs` | 50 | 100 | Stage 2 needs more time for fine-tuning |
| `patience` | 20 | 30 | Stage 2 allows more exploration |

### Expected Results

| Stage | Expected mAP@0.5 | Notes |
|-------|------------------|-------|
| Stage 1 (Feature Extraction) | ~72-75% | Head-only training, limited capacity |
| Stage 2 (Fine-Tuning) | ~78-82% | Neck adaptation improves multi-scale detection |
| From Scratch (no transfer) | ~60-65% | Would need much more data |

### Weights Location
```
runs/transfer_learning/
├── stage1_feature_extraction/
│   ├── weights/best.pt    ← Stage 1 best weights
│   └── stage1_results.json
└── stage2_fine_tuning/
    ├── weights/best.pt    ← Stage 2 best weights (FINAL)
    └── stage2_results.json
```